# IFLS5 — Weighted Analysis
Survey-weighted logistic regression using `pwt14xa` (person cross-sectional weight, IFLS5 2014).

In [ ]:
import pandas as pd
import numpy as np
import patsy
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [ ]:
def load_stata(path, cols):
    """Load Stata file; strip Categorical wrapper so columns are plain Python objects."""
    df = pd.read_stata(path, columns=cols)
    for col in df.columns:
        if pd.api.types.is_categorical_dtype(df[col]):
            df[col] = df[col].astype(object)
    return df

cov    = load_stata('adult_a/b3a_cov.dta',       ['pidlink', 'marstat', 'age', 'sex'])
dl1    = load_stata('adult_a/b3a_dl1.dta',       ['pidlink', 'dl01f', 'dl06'])
tk2    = load_stata('adult_a/b3a_tk2.dta',       ['pidlink', 'tk25a1', 'tk25a2'])
ptrack = load_stata('tracking_book/ptrack.dta',  ['pidlink', 'pwt14xa'])

print('cov   :', cov.shape)
print('dl1   :', dl1.shape)
print('tk2   :', tk2.shape)
print('ptrack:', ptrack.shape)

## 2. Merge Datasets

In [ ]:
df = cov.merge(dl1, on='pidlink', how='inner') \
        .merge(tk2, on='pidlink', how='inner') \
        .merge(ptrack, on='pidlink', how='left')

df = df.rename(columns={
    'tk25a1': 'monthly_income',
    'tk25a2': 'yearly_income',
    'dl01f' : 'ethnicity',
    'dl06'  : 'education',
})

print('Merged shape:', df.shape)
print('Weight non-null:', df['pwt14xa'].notna().sum(), '/', len(df))

## 3. Clean Variables

In [ ]:
# ── Age: remove 'Don't Know', convert to numeric, keep 18+ ───────────
df = df[df['age'].astype(str) != "998:Don't Know"].copy()
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df[df['age'] >= 18]

# ── Marital status ────────────────────────────────────────────────────
marstat_map = {
    '1:Not yet married': 'Never Married',
    '2:Married'        : 'Married',
    '3:Separated'      : 'Separated',
    '4:Divorced'       : 'Divorced',
    '5:Widowed'        : 'Widowed',
    '6:Cohabitate'     : 'Cohabitate',
}
df['marstat'] = df['marstat'].astype(str).map(marstat_map)
df = df[df['marstat'].notna()]

# ── Sex ───────────────────────────────────────────────────────────────
df['sex'] = df['sex'].astype(str).map({'1:Male': 'Male', '3:Female': 'Female'})
df = df[df['sex'].isin(['Male', 'Female'])]

# ── Ethnicity ─────────────────────────────────────────────────────────
ethnic_map = {
    'A':'Jawa','B':'Sunda','C':'Bali','D':'Batak','E':'Bugis','F':'Tionghoa',
    'G':'Madura','H':'Sasak','I':'Minang','J':'Banjar','K':'Bima-Dompu',
    'L':'Makassar','M':'Nias','N':'Palembang','O':'Sumbawa','P':'Toraja',
    'Q':'Betawi','R':'Dayak','S':'Melayu','T':'Komering','U':'Ambon',
    'A1':'Manado','B1':'Aceh','C1':'Other South Sumatera','D1':'Banten',
    'E1':'Cirebon','F1':'Gorontalo','G1':'Kutai','V':'Other',
}
df['ethnicity'] = df['ethnicity'].astype(str).map(ethnic_map).fillna('Other')

# ── Education ─────────────────────────────────────────────────────────
df['education'] = pd.to_numeric(df['education'], errors='coerce')

print('Shape after cleaning:', df.shape)
print('\nMarstat:\n', df['marstat'].value_counts())
print('\nSex:\n', df['sex'].value_counts())

## 4. Income Transform & Derived Variables

In [ ]:
# Replace IFLS missing codes
IFLS_MISSING = [999999997, 999999998, 999999999]
df['monthly_income'] = pd.to_numeric(df['monthly_income'], errors='coerce').replace(IFLS_MISSING, np.nan)
df['yearly_income']  = pd.to_numeric(df['yearly_income'],  errors='coerce').replace(IFLS_MISSING, np.nan)

df['log_income']  = np.log1p(df['monthly_income'])
df['age_squared'] = df['age'] ** 2

print(df[['monthly_income', 'log_income', 'age', 'age_squared']].describe())

## 5. Outcome & Dummy Variables

In [ ]:
df['ever_married'] = (df['marstat'] != 'Never Married').astype(int)
df['is_divorced']  = df['marstat'].isin(['Divorced', 'Separated']).astype(int)
df['is_jawa']      = (df['ethnicity'] == 'Jawa').astype(int)
df['is_sunda']     = (df['ethnicity'] == 'Sunda').astype(int)

print('ever_married:', df['ever_married'].value_counts().to_dict())
print('is_divorced :', df['is_divorced'].value_counts().to_dict())
print('sex         :', df['sex'].value_counts().to_dict())

## 6. Final Sample — Drop Missing & Save

In [ ]:
keep = ['pidlink','sex','age','age_squared','education',
        'monthly_income','log_income','marstat',
        'ever_married','is_divorced','is_jawa','is_sunda','pwt14xa']

df_clean = df[keep].dropna().reset_index(drop=True)

print('Final sample :', df_clean.shape)
print('Male         :', (df_clean['sex'] == 'Male').sum())
print('Female       :', (df_clean['sex'] == 'Female').sum())
print('Weight range :', df_clean['pwt14xa'].min(), '—', df_clean['pwt14xa'].max())

df_clean.to_csv('clean_data/data_clean_weighted.csv', index=False)

## 7. Weighted Descriptive Statistics (Table 1)

In [ ]:
def weighted_mean(series, weights):
    mask = series.notna() & weights.notna()
    return np.average(series[mask], weights=weights[mask])

def weighted_se(series, weights):
    mask = series.notna() & weights.notna()
    w, x = weights[mask].values, series[mask].values
    wm = np.average(x, weights=w)
    return np.sqrt(np.average((x - wm) ** 2, weights=w) / mask.sum())

vars_desc = {
    'Age'                  : 'age',
    'Monthly Income (IDR)' : 'monthly_income',
    'Education (years)'    : 'education',
    'Ever Married (prop)'  : 'ever_married',
    'Ever Divorced (prop)' : 'is_divorced',
    'Javanese (prop)'      : 'is_jawa',
    'Sundanese (prop)'     : 'is_sunda',
}

rows = []
for label, col in vars_desc.items():
    for grp, gdf in [('Women', df_clean[df_clean['sex'] == 'Female']),
                     ('Men',   df_clean[df_clean['sex'] == 'Male'])]:
        rows.append({
            'Variable': label, 'Group': grp,
            'Mean': round(weighted_mean(gdf[col], gdf['pwt14xa']), 3),
            'SE'  : round(weighted_se(gdf[col],   gdf['pwt14xa']), 3),
        })

long  = pd.DataFrame(rows)
tbl1  = long.pivot(index='Variable', columns='Group', values=['Mean','SE'])
tbl1.columns = [f'{stat} {grp}' for stat, grp in tbl1.columns]
tbl1  = tbl1[['Mean Women','SE Women','Mean Men','SE Men']]

print('\n=== Table 1: Weighted Means and Standard Errors ===')
print(tbl1.to_string())

## 8. Weighted Logistic Regression — Helper Functions

In [ ]:
def run_weighted_model(df_sub, outcome, formula_rhs, label):
    """
    Weighted logistic regression with survey probability weights (pwt14xa).
    Weights are normalized to sum to N before being passed as freq_weights.
    """
    pred_cols = [c.strip() for c in formula_rhs.replace('~', '').split('+')]
    all_cols  = [outcome] + pred_cols + ['pwt14xa']
    df_m = df_sub.dropna(subset=all_cols).copy()
    n = len(df_m)

    # Normalize: scale weights so they sum to sample size N
    w = df_m['pwt14xa'].values
    w_norm = w / w.sum() * n

    # patsy builds the design matrix and preserves column names
    y, X = patsy.dmatrices(f'{outcome} {formula_rhs}', data=df_m, return_type='dataframe')

    model = sm.Logit(y.values.ravel(), X, freq_weights=w_norm).fit(disp=False)

    pr2 = 1 - model.llf / model.llnull
    result_df = pd.DataFrame({
        'Coef.'   : model.params,
        'Std.Err.': model.bse,
        'z'       : model.tvalues,
        'P>|z|'   : model.pvalues,
    }, index=X.columns)

    print(f'\n--- {label} (N={n:,}) ---')
    print(result_df.round(3).to_string())
    print(f'Pseudo R²: {pr2:.4f}')
    return model, X.columns.tolist()


def format_results_table(m_female, cols_f, m_male, cols_m, title):
    """Side-by-side coefficient table with significance stars."""
    results = {}
    for label, model, cols in [('Women', m_female, cols_f), ('Men', m_male, cols_m)]:
        results[label] = {}
        for i, col in enumerate(cols):
            coef  = model.params[i]
            se    = model.bse[i]
            pval  = model.pvalues[i]
            stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
            results[label][col] = f'{coef:.3f}{stars}\n({se:.3f})'

    tbl = pd.DataFrame(results)
    pr2_w = 1 - m_female.llf / m_female.llnull
    pr2_m = 1 - m_male.llf   / m_male.llnull
    tbl.loc['Pseudo R²'] = [f'{pr2_w:.3f}', f'{pr2_m:.3f}']
    tbl.loc['N']         = [f'{int(m_female.nobs):,}', f'{int(m_male.nobs):,}']

    print(f'\n=== {title} ===')
    print(tbl.to_string())
    print('\n* p<0.05  ** p<0.01  *** p<0.001')
    return tbl

## 9. Split by Sex

In [ ]:
df_male   = df_clean[df_clean['sex'] == 'Male'].copy()
df_female = df_clean[df_clean['sex'] == 'Female'].copy()
print('Male  :', len(df_male))
print('Female:', len(df_female))

## 10. Model 1 — Ever Married

In [ ]:
formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m1_f, cols1_f = run_weighted_model(df_female, 'ever_married', formula_base, 'Ever Married — Women')
m1_m, cols1_m = run_weighted_model(df_male,   'ever_married', formula_base, 'Ever Married — Men')

tbl_m1 = format_results_table(m1_f, cols1_f, m1_m, cols1_m, 'TABLE 2: EVER MARRIED')

## 11. Model 2 — Ever Divorced (Ever Married Subsample)

In [ ]:
df_em    = df_clean[df_clean['ever_married'] == 1].copy()
df_em_f  = df_em[df_em['sex'] == 'Female'].copy()
df_em_m  = df_em[df_em['sex'] == 'Male'].copy()

print('Ever married — Female:', len(df_em_f), '| Divorced:', df_em_f['is_divorced'].sum())
print('Ever married — Male  :', len(df_em_m), '| Divorced:', df_em_m['is_divorced'].sum())

formula_div = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m2_f, cols2_f = run_weighted_model(df_em_f, 'is_divorced', formula_div, 'Ever Divorced — Women')
m2_m, cols2_m = run_weighted_model(df_em_m, 'is_divorced', formula_div, 'Ever Divorced — Men')

tbl_m2 = format_results_table(m2_f, cols2_f, m2_m, cols2_m, 'TABLE 3: EVER DIVORCED (Ever Married Only)')

## 12. Predicted Probability Plots

In [ ]:
def plot_predicted_prob(m_f, m_m, df_f, df_m, pred_cols, outcome_label, filename):
    income_range = np.linspace(
        df_clean['log_income'].quantile(0.05),
        df_clean['log_income'].quantile(0.95), 100
    )
    other_cols = [c for c in pred_cols if c not in ('Intercept', 'log_income')]

    fig, ax = plt.subplots(figsize=(9, 5))
    for model, df_sub, color, label in [
        (m_f, df_f, '#e74c3c', 'Women'),
        (m_m, df_m, '#2980b9', 'Men'),
    ]:
        means = df_sub[other_cols].mean()
        pred_df = pd.DataFrame({'log_income': income_range,
                                **{c: means[c] for c in other_cols}})
        pred_df.insert(0, 'Intercept', 1.0)
        X_pred = pred_df[pred_cols]
        prob = model.predict(X_pred.values)
        ax.plot(income_range, prob, color=color, label=label, linewidth=2)

    ax.set_xlabel('Log Monthly Income', fontsize=12)
    ax.set_ylabel('Predicted Probability', fontsize=12)
    ax.set_title(f'Predicted Probability: {outcome_label}', fontsize=13)
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()


plot_predicted_prob(m1_f, m1_m, df_female, df_male, cols1_f,
                   'Ever Married by Log Income', 'ever_married_weighted.png')

plot_predicted_prob(m2_f, m2_m, df_em_f, df_em_m, cols2_f,
                   'Ever Divorced by Log Income (Ever Married Only)', 'ever_divorced_weighted.png')

## End